# Analyse the output of a retrieval

Interactive notebook to look at the walkers of a retrieval, choose which ones to keep, and generate models,
profiles and spectra from them. Retrievals are slow to run, so this is where you play with the chains to isolate what
you think is the real signal.

**It adapts to the run** (high-res only, low-res only, joint; with or without photometry...). The cell that
loads the retrieval prints what was detected, and the sections that need something the run does not have are skipped.

**Optional sections** (slow, or only useful for specific checks) have a flag at the top of their section
(`show_..._high`, `compare_line_opacities`...), `False` by default.

Multi-region models have their own notebook (`analyse_retrieval_multi_region.ipynb`): here the profiles are shown
for the first region only.

Sections: 1. Load the retrieval, 2. Walkers and selection, 3. Model for specific parameters (and save it for
`logl_grid`), 4. Comparison with low-resolution data, 5. Contribution of each species, 6. Confidence intervals:
profiles, 7. Confidence intervals: spectra, 8. Abundance ratios *(coming)*.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from starships import retrieval
from starships import retrieval_analysis as ra
import starships.retrieval_utils as ru
from starships.plotting_fcts import (get_plot_labels, scatterplot_logl, get_plot_limits_from_data,
                                     plot_tp_sample, plot_p_profile_sample, plot_profile_combined,
                                     setup_default_plot_params, plot_spectra_sample_GTC, plot_contributions, colorsDict)
from starships import petitradtrans_utils

petitradtrans_utils.log.setLevel('INFO')

# Load the retrieval

In [ ]:
# The yaml file of each run is saved in the retrieval_params directory (scratch), you can take any of them.
yaml_file = Path.home() / 'scratch/starships_scratch/retrieval_analysis_tests' / 'yaml/params_emission_JR_g395H_1_g395H_2_modif_disso_1150020_1.yaml'
# yaml_file = Path.home() / 'scratch/starships_scratch/retrieval_analysis_tests' / 'yaml/params_emission_LRR_wfc3_g395H_1_g395H_2_spitzer_modif_disso_1200457.yaml'

In [ ]:
# This loads all the setup of the retrieval (priors, data, instruments...) but does not run anything.
retrieval.setup_retrieval(yaml_file)

# What does this run allow us to do? The sections that need something the run does not have are skipped.
run = ra.describe_run(retrieval)
print(run.summary())

# Get labels
labels = get_plot_labels(params=retrieval.params_prior)

# Get walker output path
walker_path = retrieval.walker_path

print('\nDefault retrieval walker file:', retrieval.walker_file_out)

Note that the default walker file may not be a valid file (e.g. if it was set to None in the input yaml file, or the number at the end of the name). **The retrieval `walker_file` can be changed as long as it has the same parameters.**

In [ ]:
# Show the last walker files
walker_files_recent = sorted(walker_path.glob('*.h5'), key=lambda f: f.stat().st_mtime, reverse=True)
for f in walker_files_recent[:50]:
    print(f.name)

In [ ]:
if run.is_multi_region:
    print('WARNING: multi-region run. Profiles below are for the first region only. '
          'See analyse_retrieval_multi_region.ipynb for the comparison between regions.')

## Read the walker file

In [ ]:
walker_file = Path('walker_steps_emission_JR_g395H_1_g395H_2_modif_disso_1149952_1_1_0.h5')
# walker_file = retrieval.walker_file_out

# Number of steps to discard at the start of each walker (burn-in)
discard = 0

print('walker file:', walker_file)
chains = ru.read_walkers_file(walker_path / walker_file, discard=discard)

In [ ]:
chains.shape

In [ ]:
maxs, errors = ru.find_dist_maxs(chains, labels, bin_size=6, print_latex_style=True, alpha_plot=0.05)

In [ ]:
maxs2s, errors2s = ru.find_dist_maxs(chains, labels, bin_size=6, prob=0.954, plot=False, print_latex_style=True)

In [ ]:
fig, flat_samples = ru.plot_corner(chains, labels, param_no_zero=4,
                                   maxs=maxs, errors=errors, plot_datapoints=False)

In [ ]:
# Get the posterior distribution ordered by their logl value (last value is the highest logl)
ord_pos, ord_logl, autocorr = ra.load_posterior_sample(walker_path / walker_file, discard=discard)

In [ ]:
autocorr

## Verify the logl of each parameter
**This is useful if you want to do some analysis on burnin results.**

Based on the logl scatter plot (see below):
- For burnins, are there 2 clear blobs separated vertically (in logl value)?
- Is there an actual maximum?
- Are some walkers stuck in a secondary maximum (if so, we might remove them)?
- Do you want to restrain the parameters to a subsample for later use? Like to generate the spectra.

In [ ]:
print('Median logl value:', np.median(ord_logl))

prior_keys = list(retrieval.params_prior.keys())

In [ ]:
# Make a selection of the walkers if wanted

# On the logl (Set it to -np.inf if not used)
cond = ord_logl > 124055.  # about the median of this burn-in file

# Or additional criteria on the parameters (uncomment and edit what you need).
# NOTE: for a parameter with a log-uniform prior (like the abundances), the value in the chains is its log10.
# idx = prior_keys.index('tp_gamma')
# cond &= ord_pos[:, idx] > 0.2

# For example on molecules
# idx = prior_keys.index('ptrans')
# cond &= ord_pos[:, idx] > -4.

# idx = prior_keys.index('log_f_wfc3')
# cond &= ord_pos[:, idx] < 1.5

# idx = prior_keys.index('kp')
# cond &= ord_pos[:, idx] < 200
# cond &= ord_pos[:, idx] > 190

# idx = prior_keys.index('rv')
# cond &= ord_pos[:, idx] > 10
# cond &= ord_pos[:, idx] < 20

ord_pos_sub = ord_pos[cond]
ord_logl_sub = ord_logl[cond]
print(f'{cond.sum()} / {len(cond)} samples kept')

In [ ]:
fig, ax = scatterplot_logl(ord_pos, ord_logl, n_max_pts=10000, tight_ylim=True,
                           alpha=0.1, color='k', ylabels=labels)

# Comment/uncomment the following lines to plot the sub_samples
_ = scatterplot_logl(ord_pos_sub, ord_logl_sub, n_max_pts=10000, tight_ylim=True,
                     alpha=0.3, color='darkgreen', ylabels=labels, ax=ax)

# Zoom on one parameter if needed (the index is the parameter position in the list above)
# ax[2].set_xlim(-30, 30)

### You can also remake the corner plot for the subsample

In [ ]:
fig, flat_samples = ru.plot_corner(ord_pos_sub[:, None, :], labels, param_no_zero=4, plot_datapoints=False)

## Optional: compare with a CMA-ES best-fit (`run_starships_optimize`)
If you ran `run_starships_optimize` on this burn-in, its `.npz` result(s) hold a genuine
maximum-a-posteriori fit (only the maximum, no exploration of the rest of the posterior -- see
`starships/retrieval_optimize.py`'s module docstring), which can be a real amount higher than any
MCMC sample, plus every candidate CMA-ES evaluated along the way (`theta_trace`/`lnprob_trace`).

The dedicated notebook `tutorials/retrieval_optimize_examples/Inspect_optimize_trace.ipynb`
compares several starts against each other (no petitRADTRANS needed there, faster to iterate on) --
do that first if you ran more than one start, to check they agree before trusting one. Here, once
you know which one to trust, this section compares it to the MCMC chain above and lets you use its
best-fit for the model further down. **Optional**: leave `use_optimizer_result = False` to skip it.

In [ ]:
use_optimizer_result = False

# --- Edit this: the .npz result files, one per CMA-ES start ---
optimizer_result_files = [walker_path / f'optimum_{walker_file.stem}_start{i}.npz' for i in (1, 2, 3)]

opt_result = None
if use_optimizer_result:
    from starships.retrieval_optimize import compare_results

    compare_results(optimizer_result_files)  # same table as run_starships_optimize --compare

    # Picks the start with the highest lnprob_best -- edit to force a specific one instead
    # (e.g. chosen_file = optimizer_result_files[0]) if the comparison above suggests otherwise:
    # starts far from each other in lnprob mean the maximum is not reliable, see the printout.
    chosen_file = max(optimizer_result_files, key=lambda f: np.load(f)['lnprob_best'])
    opt_result = np.load(chosen_file)
    print(f'Using {chosen_file.name}')

    if list(opt_result['param_names']) != prior_keys:
        print('WARNING: parameter order/names differ between this yaml and the optimizer result -- '
             'best_params below would be wrong. Check you loaded the matching yaml file.')

In [ ]:
if opt_result is not None:
    n_trace, n_evals = len(opt_result['lnprob_trace']), int(opt_result['n_evals'])
    print(f"lnprob: burn-in best = {ord_logl.max():.2f}  |  optimizer start = "
         f"{float(opt_result['lnprob_start']):.2f}  |  optimizer best = {float(opt_result['lnprob_best']):.2f}")
    print(f"Trace covers {n_trace}/{n_evals} evaluations ({100 * n_trace / max(n_evals, 1):.0f}%) -- "
         "less than 100% for an old run resumed after the trace feature was added, see the quicklook "
         "notebook mentioned above.")

    # Convergence curve, with the MCMC's own best logl as a reference
    fig, ax = plt.subplots(figsize=(8, 4))
    history = opt_result['history']  # columns: n_evals, best lnprob so far, generation median, sigma
    ax.plot(history[:, 0], history[:, 1], label='Best so far (CMA-ES)')
    ax.plot(history[:, 0], history[:, 2], alpha=0.5, label='Generation median')
    ax.axhline(ord_logl.max(), color='k', linestyle='--', label='Best MCMC sample (this file)')
    ax.axhline(float(opt_result['lnprob_start']), color='grey', linestyle=':', label='Burn-in start point')
    ax.set_xlabel('Number of lnprob evaluations')
    ax.set_ylabel('lnprob')
    ax.legend(fontsize='small')

In [ ]:
if opt_result is not None:
    # Overlay the optimizer's search trace on the MCMC's own per-parameter logl scatter plot
    # (same scatterplot_logl tool the quicklook notebook uses to compare several starts to each
    # other, applied here instead to compare one start with the chain).
    fig, ax = scatterplot_logl(ord_pos, ord_logl, n_max_pts=10000, tight_ylim=True,
                              alpha=0.1, color='k', ylabels=labels)
    _ = scatterplot_logl(opt_result['theta_trace'], opt_result['lnprob_trace'], n_max_pts=10000,
                        tight_ylim=False, alpha=0.15, color='crimson', ylabels=labels, ax=ax)
    for ax_i, best_val in zip(ax, opt_result['theta_best']):
        ax_i.axvline(best_val, color='crimson', linestyle='--', linewidth=1)
    fig.suptitle('Black: MCMC chain -- Red: CMA-ES trace (dashed line = best-fit)', y=1.01)

# Change stellar spectrum to blackbody if wanted
Usually you don't want to do this, but it could be useful to differentiate the stellar signal from the planet's.
I did it personally to make plots for a presentation. If your stellar model is already a blackbody, then don't bother.

In [ ]:
from astropy.modeling import models
import astropy.units as u

# Use the temperature corresponding to the stellar spectrum (you can also play with it)
teff = 12000

if retrieval.star_spectrum is not None:

    fct_star = models.BlackBody(temperature=teff*u.K, scale=1*u.Unit('erg / (cm ** 2 * s * AA * sr)'))

    # Here I use a normalisation factor to make sure both models are agreeing.
    # The wavelength range is the range from WFC3 (the stellar spectrum is most important in spectrophotometry)
    wv = retrieval.star_wv
    in_range = (1.1 < wv) & (wv < 1.6)
    norm_factor = np.mean(fct_star(wv[in_range]*u.um)) / np.mean(retrieval.star_flux[in_range])

    def fct_star_BB(wv):
        return fct_star(wv*u.um) / norm_factor

    plt.plot(retrieval.star_wv[in_range], fct_star_BB(retrieval.star_wv)[in_range], label=f"scaled BB({teff}K)")

    plt.plot(retrieval.star_wv[in_range], retrieval.star_flux[in_range])
    plt.legend()

### Change the flux or not (comment if not)

In [ ]:
# retrieval.star_flux = fct_star_BB(retrieval.star_wv)

# Show the model for specific parameters

## Generate/initialize the model

### Choose which parameters you want (best logL?, median?)
TIP: Look at the logL plot to see which one represents the best your results.

In [ ]:
params_set = ord_pos_sub  # Sub sample defined above
# params_set = ord_pos  # full sample

# best_params = params_set[-1]  # Best logl
best_params = params_set[len(params_set) // 2]  # Median
# best_params = opt_result['theta_best']  # Use the CMA-ES best-fit instead (see the optional section above)

print(best_params)

### Convert the parameters array to a dictionary of parameters
`theta_regions` is a list of dictionaries with all the potential parameters that could be useful.
You can edit it here if you want before generating the model. Each dictionary represents a specific region
on the planet (so usually just one region).

In [ ]:
theta_regions = retrieval.unpack_theta(best_params)

In [ ]:
# Print the fitted parameters of each region
for region, theta_dict in zip(run.region_ids, theta_regions):
    print(f'--- Region {region}')
    for key in retrieval.params_prior:
        if key in theta_dict:
            print(key, theta_dict[key])

### Compute the spectrum (the first time is slower due to initialization)
NOTE: The low-resolution model uses the correlated-k opacities (c-k), used for the fit of low-resolution data
(spectrophotometry, photometry). The high-resolution model uses line-by-line opacities (lbl): it is what the
high-resolution data was fitted with, and it is also used for the low-resolution instruments that were set to
`opacity_mode: 'lbl'`. Both are shown to make sure they are consistent.

The lbl model can be computed whenever `run.can_model_high` is true, **even for a low-resolution run** (the yaml file
has a high-resolution `instrum` or an lbl instrument). It is slow the first time (the line-by-line opacities are loaded),
so it is off by default for low-resolution runs: set `compute_high_res_model = True` if you want it.

In [ ]:
compute_high_res_model = run.can_model_high and run.has_high_res_data
# compute_high_res_model = True

wv_low, spec_low, _ = retrieval.prepare_static_model(theta_regions, 'low')

wv_high, spec_high = None, None
if compute_high_res_model:
    wv_high, spec_high, _ = retrieval.prepare_static_model(theta_regions, 'high')

In [ ]:
if spec_high is not None:
    plt.plot(wv_high, spec_high * 1e6, label='lbl')
plt.plot(wv_low, spec_low * 1e6, label='c-k')
plt.xlabel('Wavelength [um]')
plt.ylabel(f'{retrieval.kind_trans} spectrum [ppm]')
plt.legend()

### Save the model to give to `logl_grid`
The saved file has the native (line-by-line) planet and star spectra of each region and the parameters, so
`logl_grid` can redo exactly what the retrieval does (resolution of each visit, rotation kernels...).
It only makes sense for runs with high-resolution data. It uses `theta_regions` (not `best_params`), so the
edits you made above are saved. Give the file a name that says what selection it comes from.

In [ ]:
save_model_for_logl_grid = False
dir_models = Path.home() / 'scratch/starships_scratch/retrieval_analysis_tests/tmp_out'
model_file = dir_models / 'kelt20b_JR_test_model.npz'

if save_model_for_logl_grid:
    if not run.has_high_res_data:
        print('This run has no high-resolution data: there is nothing to give to logl_grid.')
    else:
        dir_models.mkdir(parents=True, exist_ok=True)
        planet_model = retrieval.save_planet_model(theta_regions, model_file)
        print(planet_model.describe())

### Show/save the spectrum for each individual species (line opacity) -- optional, slow
One line-by-line model per species in `retrieval.line_opacities`, all the *other* line species set to
~0 -- the continuum opacities (`retrieval.continuum_opacities`, e.g. H-, e-) and `other_species` are
NOT touched, so "H2O only" really means "H2O on top of the continuum", not "nothing but H2O". Can be
useful to perform cross-correlations for example, and/or to save one model per species for `logl_grid`
alongside the main "all species" model saved above (same file naming: `<model file stem>_<species>_only.npz`).

In [ ]:
show_individual_species_high = False
save_individual_species_models = False

if (show_individual_species_high or save_individual_species_models) and run.can_model_high:
    for single_mol in retrieval.line_opacities:

        # Copy input dict
        theta_reg_single = [dict(theta_dict) for theta_dict in theta_regions]

        # Set all other linelists to zero (10^-99 to avoid division by zero). Continuum opacities
        # and other_species are untouched: the loop only ever assigns keys in line_opacities.
        for theta_dict in theta_reg_single:
            for mol in retrieval.line_opacities:
                if mol != single_mol:
                    theta_dict[mol] = 1e-99

        if show_individual_species_high:
            wv_i, spec_i, _ = retrieval.prepare_static_model(theta_reg_single, 'high')
            plt.plot(wv_i, spec_i, label=f'{single_mol} only', alpha=0.3)

        if save_individual_species_models:
            dir_models.mkdir(parents=True, exist_ok=True)
            species_model_file = dir_models / f'{model_file.stem}_{single_mol}_only.npz'
            retrieval.save_planet_model(theta_reg_single, species_model_file)
            print(f'Saved {species_model_file}')

    if show_individual_species_high:
        plt.legend()

## Quick comparison with low-resolution data
All the data available in the yaml file is plotted, but it may not have been included in the fit
(e.g. if you were using HRR mode). Nothing to compare (only the model is shown) if the run has no
spectrophotometric or photometric data.

In [ ]:
# Uncomment to change some settings of the low-res data before the comparison
# retrieval.spectrophotometric_data['g395H_1']['res'] = 900
# retrieval.spectrophotometric_data['g395H_2']['res'] = 900
# retrieval.photometric_data['spitzer']['wave'][1] *= 1.0001

In [ ]:
# Low-res instruments that were set to be modelled at high resolution (lbl) need the high-res model
if 'high' in run.low_res_instruments.values() and spec_high is None:
    wv_high, spec_high, _ = retrieval.prepare_static_model(theta_regions, 'high')

# Make synthetic data for all low-res instruments
synt_data_dict = ru.get_syntetic_data_low_res(wv_low, spec_low, wv_high, spec_high, retrieval_obj=retrieval)

In [ ]:
n_subplots = len(retrieval.spectrophotometric_data) + 1
fig, ax = plt.subplots(n_subplots, 1, figsize=(6, 3*n_subplots), squeeze=False)
ax = ax[:, 0]

# Plot the model in each panel and set labels
for ax_i in ax:
    if spec_high is not None:
        ax_i.plot(wv_high, spec_high * 1e6, label='lbl')
    ax_i.plot(wv_low, spec_low * 1e6, label="c-k")
    ax_i.set_ylabel(f"{retrieval.kind_trans} spectrum [ppm]")

# Make one panel per spectrophotometric instrument name
for i_instru, (instru_name, infos) in enumerate(retrieval.spectrophotometric_data.items()):

    x_plot, y_plot, yerr = infos['wave'], infos['data'], infos['err']
    y_plot = y_plot * 1e6  # in ppm
    yerr = yerr * 1e6  # in ppm

    # Force the plot limits around the low-res data
    xlim = get_plot_limits_from_data(x_plot)
    ylim = get_plot_limits_from_data(y_plot, pad=0.2)

    # Plots
    ax[i_instru].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", label=instru_name)
    ax[i_instru].plot(x_plot, synt_data_dict[instru_name]*1e6, '.', label='Model binned')
    ax[i_instru].set_xlim(xlim)
    ax[i_instru].set_ylim(ylim)

    # Also plot in the last panel
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", label=instru_name)

    # Labels
    ax[i_instru].set_title(f"Zoom on {instru_name} data", loc='left')
    ax[i_instru].legend()


# Put everything in the last panel
for instru_name, infos in retrieval.photometric_data.items():

    x_plot, y_plot, yerr, x_limits = (infos[key] for key in ['wave', 'data', 'err', 'wv_coverages'])
    y_plot = y_plot * 1e6  # in ppm
    yerr = yerr * 1e6  # in ppm

    xerr = np.abs(np.transpose(x_limits) - x_plot)
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, xerr=xerr, fmt=".", label=instru_name)
    ax[-1].plot(x_plot, synt_data_dict[instru_name]*1e6, '.', label=f'Synt data ({instru_name})')


ax[-1].set_title(f"All data", loc='left')
ax[-1].set_xlabel("Wavelength [um]")
ax[-1].legend()
plt.tight_layout()

## Isolate the contribution of each species

In [ ]:
specie_list = retrieval.line_opacities  # ['H2O', 'TiO', 'VO', 'FeH', 'OH', 'CO', 'H-']
wv, spec_dict = ru.get_contribution(specie_list, retrieval, theta_regions, mode='low')

In [ ]:
fig, ax = plot_contributions(wv, spec_dict, retrieval.kind_trans, figsize=(16, 32))

### Same at high resolution (line by line) -- optional, slow
Each entry of `specie_list_high` is a group of species kept together (the others are removed).

In [ ]:
show_contribution_high = False
specie_list_high = [['Fe', 'H2O', 'OH', 'H-'], ['Fe', 'H2O', 'H-'], ['Fe', 'OH', 'H-'], ['OH', 'H2O', 'H-']]

if show_contribution_high and run.can_model_high:
    wv_h, spec_dict_h = ru.get_contribution(specie_list_high, retrieval, theta_regions, mode='high')
    fig, ax = plot_contributions(wv_h, spec_dict_h, retrieval.kind_trans, figsize=(8, 16))

## Compare line opacities (low VS high) -- optional, slow
**This is a good check to make when doing Joint Retrievals (JR)**, but generating the high-res spectra can take a
lot of time, so don't do this every time. It needs both models (`run.can_model_high`).

In [ ]:
compare_line_opacities = False

if compare_line_opacities and run.can_model_high:
    specie_list = [[mol, 'H-'] for mol in retrieval.line_opacities] + [['H-']]
    wv_high_c, spec_dict_high = ru.get_contribution(specie_list, retrieval, theta_regions, mode='high')
    wv_low_c, spec_dict_low = ru.get_contribution(specie_list, retrieval, theta_regions, mode='low')

    n_species = len(specie_list)
    fig, ax = plt.subplots(n_species, 1, sharex=True, figsize=(8, n_species*2.7), squeeze=False)
    ax = ax[:, 0]

    for idx, specie in enumerate(specie_list):
        specie_label = ' + '.join(specie) if isinstance(specie, list) else specie
        ax[idx].plot(wv_high_c, spec_dict_high[f'{specie_label} only'], label=f'{specie_label} high')
        ax[idx].plot(wv_low_c, spec_dict_low['All'], label='All')
        ax[idx].plot(wv_low_c, spec_dict_low[f'{specie_label} only'], label=f'{specie_label} low')
        ax[idx].legend()

# Explore the confidence intervals
TP profiles, abundance profiles, spectra.

## Abundances and TP profiles

### Choose which sample to use (could be a subsample)
The profiles of `n_draw_profiles` random samples are computed. The abundances are computed the same way as in the
retrieval (free chemistry, with dissociation if it was included). Values are clipped to reasonable ranges so the
statistics (done in log space for the abundances) stay finite.

In [ ]:
# flatten_sample = flat_samples
# flatten_sample = chains.reshape((-1, chains.shape[-1]))
flatten_sample = ord_pos_sub

n_draw_profiles = 100

In [ ]:
# Profiles of the first region (see the multi-region notebook to compare the regions)
pressures, profile_samples_by_region = ra.draw_region_profiles(n_draw_profiles, flatten_sample, retrieval)
profile_samples = profile_samples_by_region[0]

# `profile_samples` has all the species that the model uses + the temperature profile
all_species = [key for key in profile_samples if key != 'temperature']

### Here is an example of the different profiles drawn from the sample.

In [ ]:
mol_name = 'H2O'

fig, ax = plt.subplots(1, 2, sharey=True)
ax[0].semilogy(profile_samples['temperature'].T, pressures, "k", alpha=0.1)
ax[0].invert_yaxis()

ax[1].loglog(profile_samples[mol_name].T, pressures, "k", alpha=0.2)
ax[1].set_title(mol_name)

ax[0].set_ylabel('Pressure')
ax[0].set_xlabel('Temperature')
ax[1].set_xlabel('Abundance')

### From these samples, we can get the different statistics (median, 1-sigma, 2-sigma)

In [ ]:
# Get statistics. Done in logspace for molecules, but not for temperature
profile_stats = ra.compute_profile_stats(profile_samples)

### TP profiles statistics

In [ ]:
fig, ax = plot_tp_sample(pressures, profile_stats['temperature'])

fig_file = Path('Figures/Profiles/') / Path(f'{walker_file.stem}_zoom.png')
# fig.savefig(fig_file)

### Abundance profiles statistics

In [ ]:
# Species shown in each panel (only the ones the model has)
species_panels = [['H2O', 'H-', 'OH'], ['H-', 'e-'], ['H', 'H2']]

fig, ax = plt.subplots(1, 1 + len(species_panels), sharey=True, figsize=(10, 4))

plot_tp_sample(pressures, profile_stats['temperature'], ax=ax[0])

for ax_i, mol_names in zip(ax[1:], species_panels):
    for mol_name in mol_names:
        if mol_name in profile_stats:
            plot_p_profile_sample(pressures, profile_stats[mol_name], ax=ax_i, label=mol_name)
    ax_i.set_xscale('log')
    ax_i.set_xlim(1e-12, 1e0)
    ax_i.legend()

ax[2].set_xlabel('Volume Mixing Ratio', fontsize='large')

### You can also overplot what is expected from chemical equilibrium
Needs petitRADTRANS. `c_to_o` is fixed by hand, and the metallicity is estimated from the free-chemistry Fe/H
ratio if `Fe` is fitted (otherwise `metallicity`, 0 = solar). NOTE: `metallicity` here is petitRADTRANS's
equilibrium-chemistry knob, NOT literally the elemental Fe/H ratio (Fe/H is only used as its standard proxy).

In [ ]:
compare_chemical_equilibrium = True

if compare_chemical_equilibrium:
    c_to_o = .95
    metallicity = None

    profile_samples_eq = ra.equilibrium_profiles(profile_samples, pressures, c_to_o=c_to_o, metallicity=metallicity)

    # Get statistics. Done in logspace for molecules
    profile_stats_eq = ra.compute_profile_stats(profile_samples_eq)

    # Choose which specie to plot
    species_plt = [specie for specie in profile_stats_eq if profile_stats_eq[specie]['median'].max() > 1e-6]

In [ ]:
if compare_chemical_equilibrium:
    fig, ax = plt.subplots(3, 1, sharey=True, figsize=(8, 16))

    for specie in profile_stats:
        if specie in profile_stats_eq:
            plot_p_profile_sample(pressures, profile_stats[specie], ax=ax[0], label=specie)
            plot_p_profile_sample(pressures, profile_stats_eq[specie], ax=ax[1], label=specie)

    for specie in species_plt:
        if specie not in profile_stats:
            plot_p_profile_sample(pressures, profile_stats_eq[specie], ax=ax[2], label=specie)

    ax[0].set_title('Free Chemistry (shared species)', loc='left', fontsize='small')
    ax[1].set_title('Equilibrium Chemistry (shared species)', loc='left', fontsize='small')
    ax[2].set_title('Equilibrium Chemistry (not in Free Chemistry)', loc='left', fontsize='small')

    for ax_i in ax:
        ax_i.set_xscale('log')
        ax_i.set_xlim(1e-12, 1e0)
        ax_i.legend(fontsize='small')
        ax_i.set_ylabel('Pressure [bar]', fontsize='large')

    ax[2].set_xlabel('Volume Mixing Ratio', fontsize='large')

`plot_profile_combined` (now in `starships.plotting_fcts`) puts free and equilibrium chemistry on the same axes.

In [ ]:
if compare_chemical_equilibrium:
    species_compared = None  # ['CO']

    if species_compared is None:
        species_compared = [specie for specie in profile_stats if specie in profile_stats_eq]

    n_species = len(species_compared)
    fig, ax = plt.subplots(1, n_species + 1, sharey=True, figsize=(3 * n_species, 4))

    for ax_i, specie in zip(ax, species_compared):
        _, _, handles = plot_profile_combined(pressures, [profile_stats_eq[specie], profile_stats[specie]], ax=ax_i)
        ax_i.set_xscale('log')
        ax_i.set_title(specie)
        ax_i.set_ylabel(None)

    # Add a legend beside the panels
    ax[-1].axis('off')
    # Position of the text
    xpos = 0.1
    ypos = 0.9
    d_ypos = -0.1
    for handle, label in zip(handles, ['Equilibrium', 'Free']):
        color = handle.get_facecolor()[0]
        ax[-1].text(xpos, ypos, label, color=color, fontsize=16, transform=ax[-1].transAxes)
        ypos += d_ypos

    ax[0].set_ylabel('Pressure [bar]', fontsize='large')
    ax[n_species // 2].set_xlabel('Volume Mixing Ratio', fontsize='large')

# Spectra statistics

## First, we need to generate the spectra from the sample

In [ ]:
n_draw = 50
# Specific species if you want to compute their contribution (each one adds 2 spectra per draw).
# Put an empty list or None to skip
species_list = []

In [ ]:
# Generate the samples (low resolution, c-k). Use kind_res='high' for the line-by-line model (much slower).
waves, spec_samples_by_name = ra.draw_spectra(n_draw, flatten_sample, retrieval, kind_res='low',
                                              species_list=species_list)
wv, spec_samples = waves['global'], spec_samples_by_name['global']

# Get the stats
spec_stats = {key: ru.get_stats_from_profile(sp_sample) for key, sp_sample in spec_samples.items()}

## Then generate the synthetic data for each low res instrument

In [ ]:
# Instruments modelled at high resolution (lbl) need the line-by-line spectra of the same draws
spec_samples_high, wv_samples_high = None, None
if 'high' in run.low_res_instruments.values():
    waves_h, spec_samples_high_by_name = ra.draw_spectra(n_draw, flatten_sample, retrieval, kind_res='high')
    wv_samples_high, spec_samples_high = waves_h['global'], spec_samples_high_by_name['global']['All']

synt_data_samples = {key: ra.synthetic_data_samples(sp_sample, retrieval, wave_low=wv,
                                                    spec_samples_high=spec_samples_high, wave_high=wv_samples_high)
                     for key, sp_sample in spec_samples.items()}

# Get the stats
synt_data_stats = {key: {instru: ru.get_stats_from_profile(sp_sample) for instru, sp_sample in samples.items()}
                   for key, samples in synt_data_samples.items()}

## Make the retrieved spectra confidence intervals plot

In [ ]:
setup_default_plot_params()

In [ ]:
save_fig = False

specie = 'All'
scale_factor = 100  # Scale the spectrum to change units (100 -> units in percent)
ylabel = r'F$_p$ / F$_{\star}$ [\%]'
spec_stats_colors = 'greens'  # Color set from pyGTC

# You can add some plotting infos to the specific instruments (like the color!)
# 'wv_offset': To shift statistics in the plot so they don't overlap with the datapoints (better display)
instrument_plot_infos = {
    'spitzer': dict(data_color='purple', stats_color='deeppink', wv_offset=0.05),
}
for instru_name, plot_infos in instrument_plot_infos.items():
    for data_dict in [retrieval.spectrophotometric_data, retrieval.photometric_data]:
        if instru_name in data_dict:
            data_dict[instru_name].update(plot_infos)


# ==============
# Do the plot
# ==============

n_subplots = len(retrieval.spectrophotometric_data) + 1
fig, ax = plt.subplots(n_subplots, 1, figsize=(10, 5*n_subplots), squeeze=False)
ax = ax[:, 0]

# Plot the model in each panel and set labels
for ax_i in ax:
    if spec_high is not None:
        ax_i.plot(wv_high, spec_high * scale_factor, 'lightblue', label='High res', zorder=-1)
    ax_i.plot(wv_low, spec_low * scale_factor, label="Low res")
    ax_i.set_ylabel(ylabel)

# Plot the full model stats in the last panel
stats_plot = spec_stats[specie]
plot_spectra_sample_GTC(wv, [stats_plot], colorsOrder=[spec_stats_colors],
                        ax=ax[-1], scale_factor=scale_factor)

# Make one panel per spectrophotmetric instrument name
for i_instru, (instru_name, infos) in enumerate(retrieval.spectrophotometric_data.items()):

    # Get the observation
    x_plot, y_plot, yerr = infos['wave'], infos['data'], infos['err']
    y_plot = y_plot * scale_factor  # scaled to plot units
    yerr = yerr * scale_factor  # scaled to plot units

    # Get the stats
    stats_plot = synt_data_stats[specie][instru_name]

    # Force the plot limits around the low-res data
    xlim = get_plot_limits_from_data(x_plot)
    ylim = get_plot_limits_from_data(y_plot, pad=0.2)

    # Infos for the plot, specific to instrument
    data_color = infos.get('data_color', None)

    # Plots
    plot_spectra_sample_GTC(x_plot, [stats_plot], colorsOrder=[spec_stats_colors],
                            ax=ax[i_instru], scale_factor=scale_factor)
    ax[i_instru].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", color=data_color, label=instru_name)
    ax[i_instru].set_xlim(xlim)
    ax[i_instru].set_ylim(ylim)

    # Also plot the instrument data in the last panel
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", color=data_color, label=instru_name)

    # Labels
    ax[i_instru].set_title(f"Zoom on {instru_name} data", loc='left')
    ax[i_instru].legend(fontsize=12)


# Put everything in the last panel
for instru_name, infos in retrieval.photometric_data.items():

    x_plot, y_plot, yerr, x_limits = (infos[key] for key in ['wave', 'data', 'err', 'wv_coverages'])
    y_plot = y_plot * scale_factor  # scaled to plot units
    yerr = yerr * scale_factor  # scaled to plot units

    # Get the stats
    stats_plot = synt_data_stats[specie][instru_name]

    # Infos for the plot, specific to instrument
    data_color = infos.get('data_color', None)
    stats_color = infos.get('stats_color', None)
    x_offset = infos.get('wv_offset', 0.)

    # Common plotting arguments
    comm_kwargs = dict(linestyle='', marker='', linewidth=2)

    # Plot observed data
    xerr = np.abs(np.transpose(x_limits) - x_plot)
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, xerr=xerr,
                    color=data_color, label=instru_name, **comm_kwargs)

    # Plot synthetic data stats (1 and 2 sigma)
    yerr = np.abs(stats_plot['2-sig'] - stats_plot['median'][None, :])
    ax[-1].errorbar(x_plot - x_offset, stats_plot['median'] * scale_factor,
                   yerr=yerr * scale_factor, color=stats_color,
                   capsize=8, capthick=2, **comm_kwargs)

    yerr = np.abs(stats_plot['1-sig'] - stats_plot['median'][None, :])
    ax[-1].errorbar(x_plot - x_offset, stats_plot['median'] * scale_factor,
                   yerr=yerr * scale_factor, color=stats_color,
                   capsize=4, capthick=2, **comm_kwargs)

ax[-1].set_xlabel(r'Wavelength [$\mu$m]')
ax[-1].set_ylabel(ylabel)
ax[-1].legend(fontsize=12)

fig_filename = Path(walker_file.stem.replace('walker_steps', 'spectra')).with_suffix('.pdf')
if save_fig:
    fig_filename = Path('Spectra_profiles') / fig_filename
    fig_filename.parent.mkdir(exist_ok=True)
    print('Saving to', fig_filename)
    fig.savefig(fig_filename)

# Abundance ratios
Elemental abundance ratios (e.g. C/O) and metallicities relative to solar ([X/H]) computed from the same
abundance-profile sample as above (`profile_samples`, first region only). **A free-chemistry ratio does not have
to be constant with altitude**, so the comparison below is shown both as a full profile (like the TP/abundance
profiles) and at a few pressures chosen by hand -- the same ratio can look quite different depending on which
pressure is picked.

In [ ]:
# Elemental ratios (plain ratio, e.g. C/O) and metallicities (dex relative to a reference composition, e.g. Fe/H)
elemental_ratios = ['C/O']
metallicity_ratios = ['C/H', 'O/H', 'Fe/H']  # drop the ones your species don't cover (e.g. remove 'Fe/H' if Fe
                                              # isn't a fitted species -- it would just be compared to abund0=0)

# Reference composition for the metallicities: None = solar (starships.retrieval_utils.solar_asplund2009).
# Use ru.protosolar_asplund2009 for protosolar values instead.
reference_values = None

# Pressures (bar) where the ratios are compared -- pick values spanning the range you care about
pressures_compared = [1e-4, 1e-2, 1e0]

In [ ]:
# Compute the profiles (n_draw, n_pressures) and remember which ones are in dex (for the stats/plot scale)
ratio_profiles = {r: ra.abundance_ratio_profiles(profile_samples, r) for r in elemental_ratios}
ratio_profiles.update({r: ra.metallicity_profiles(profile_samples, r, reference_values=reference_values)
                       for r in metallicity_ratios})
ratio_is_dex = {r: (r in metallicity_ratios) for r in ratio_profiles}

## Ratio profiles (confidence intervals and individual draws)

In [ ]:
n_show = 50  # individual profiles drawn in the bottom row

n_ratios = len(ratio_profiles)
fig, ax = plt.subplots(2, n_ratios, sharey=True, figsize=(3.5 * n_ratios, 7), squeeze=False)

for i_col, (name, profile) in enumerate(ratio_profiles.items()):
    is_dex = ratio_is_dex[name]
    stats = ru.get_stats_from_profile(profile, log_scale=not is_dex)
    plot_p_profile_sample(pressures, stats, ax=ax[0, i_col])
    ax[1, i_col].plot(profile[:n_show].T, pressures, 'k', alpha=0.15)
    ax[1, i_col].set_yscale('log')
    ax[1, i_col].set_ylim(ax[0, i_col].get_ylim())

    for ax_i in ax[:, i_col]:
        if is_dex:
            ax_i.axvline(0., linestyle='--', color='grey')  # 0 = solar (or the chosen reference)
        else:
            ax_i.set_xscale('log')
    ax[0, i_col].set_title(name)
    ax[1, i_col].set_xlabel('[X/Y] [dex]' if is_dex else 'Ratio')

ax[0, 0].set_ylabel('Pressure [bar]')
ax[1, 0].set_ylabel('Pressure [bar]')
plt.tight_layout()

## Comparison at the chosen pressures
Median, 1-sigma and 2-sigma intervals of each ratio at each pressure of `pressures_compared` (see the cell above).
Look at how much the points move from one pressure to the next: a ratio that is far from constant with altitude
means the choice of pressure genuinely matters for any single-number comparison (e.g. to the literature).

In [ ]:
fig, ax = plt.subplots(1, n_ratios, figsize=(4 * n_ratios, 4), squeeze=False)
ax = ax[0]

print(f'{"ratio":<8} {"pressure [bar]":>15} {"median":>10} {"1-sig low":>10} {"1-sig high":>10}')
for ax_i, (name, profile) in zip(ax, ratio_profiles.items()):
    stats_by_p = ra.profile_stats_at_pressures(profile, pressures, pressures_compared,
                                               log_scale=not ratio_is_dex[name])
    xs = np.arange(len(pressures_compared))
    medians = np.array([stats_by_p[p]['median'] for p in pressures_compared])
    err1 = np.array([stats_by_p[p]['1-sig'] for p in pressures_compared])  # (n_pressures, 2) = (low, high)
    err2 = np.array([stats_by_p[p]['2-sig'] for p in pressures_compared])

    ax_i.errorbar(xs, medians, yerr=[medians - err2[:, 0], err2[:, 1] - medians],
                 fmt='none', capsize=8, capthick=2, color='C0', alpha=0.5)
    ax_i.errorbar(xs, medians, yerr=[medians - err1[:, 0], err1[:, 1] - medians],
                 fmt='o', capsize=4, capthick=2, color='C0')
    ax_i.set_xticks(xs)
    ax_i.set_xticklabels([f'{p:.0e}' for p in pressures_compared])
    ax_i.set_xlabel('Pressure [bar]')
    ax_i.set_title(name)
    if ratio_is_dex[name]:
        ax_i.axhline(0., linestyle='--', color='grey')

    for p in pressures_compared:
        s = stats_by_p[p]
        print(f'{name:<8} {p:>15.0e} {s["median"]:>10.3g} {s["1-sig"][0]:>10.3g} {s["1-sig"][1]:>10.3g}')

plt.tight_layout()

## Compare to chemical equilibrium at one pressure
Fits, for each drawn profile, the equilibrium-chemistry C/O (and metallicity) that reproduces the free-chemistry
`mol_list_eq` abundances at `press_eq` -- the equilibrium-chemistry equivalent of the ratios above, at that one
pressure. Needs petitRADTRANS. A fit failing at an extreme, ill-constrained pressure is expected sometimes, not
itself a bug -- check how many of the `n_draw_profiles` draws actually converged (printed below).

In [ ]:
compare_chemical_equilibrium_ratio = True
press_eq = 1e-2      # bar
mol_list_eq = ['H2O', 'CO']
fixed_metallicity = None  # None -> estimated from the free-chemistry Fe/H ratio of each draw if available,
                          # else solar. NOTE: this is petitRADTRANS's equilibrium metallicity knob, not
                          # literally the elemental Fe/H ratio plotted above (Fe/H is only its standard proxy).

if compare_chemical_equilibrium_ratio:
    eq_fit = ra.fit_chemical_equilibrium_at_pressure(profile_samples, pressures, press_eq,
                                                      mol_list=mol_list_eq, metallicity=fixed_metallicity)
    print(f'Free fit: {len(eq_fit["C_to_O"])}/{n_draw_profiles} draws converged')
    print(f'Fixed-metallicity fit: {len(eq_fit["C_to_O_fixed_metallicity"])}/{n_draw_profiles} draws converged')

    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].hist(eq_fit['C_to_O'], bins=20, alpha=0.5, label='Metallicity free')
    ax[0].hist(eq_fit['C_to_O_fixed_metallicity'], bins=20, alpha=0.5, label='Metallicity fixed')
    ax[0].axvline(np.median(ratio_profiles['C/O'][:, np.searchsorted(pressures, press_eq)]), color='k',
                 linestyle='--', label='Free chemistry C/O')
    ax[0].set_xlabel('Equilibrium C/O')
    ax[0].legend(fontsize='small')

    ax[1].hist(eq_fit['metallicity'], bins=20, color='C2')
    ax[1].set_xlabel('Equilibrium metallicity [dex]')
    plt.tight_layout()